# Обучение ResNet18 для классификации AI vs Human Generated Images

Интеграция TensorBoard, S3 (MinIO), DVC-пайплайн и сравнение версий.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt
import boto3
from botocore.client import Config
import json
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

c:\Users\gfgh\Desktop\HSE\VK_DL\ipynb_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.0+cu118
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Using device: cuda


In [2]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path = self._fix_image_issues(self.data.iloc[idx]['file_name'])
        image = Image.open(img_path).convert('RGB')
        label = self.data.iloc[idx]['label']
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

    def _fix_image_issues(self, file_name: str) -> Path:
        file = self.root_dir / file_name
        if file.exists():
            return file
        file = self.root_dir / file_name.replace("train_data/", "test_data/", 1)
        if file.exists():
            return file
        raise FileNotFoundError(f"Image not found for {file_name!r} under {self.root_dir}")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset-hw/Train_1/train.csv',
    root_dir='ai-vs-human-generated-dataset-hw/Train_1',
    transform=train_transform
)

test_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset-hw/Test_1/test.csv',
    root_dir='ai-vs-human-generated-dataset-hw/Test_1',
    transform=test_transform
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

Train dataset size: 9993
Test dataset size: 3997


Добавим нашу модель!

In [3]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)
print(model)

c:\Users\gfgh\Desktop\HSE\VK_DL\ipynb_venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\gfgh\Desktop\HSE\VK_DL\ipynb_venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for images, labels in tqdm(dataloader, desc='Training'):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    epoch_precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    epoch_recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    
    return epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall

Инициализируем доску и прогоним базовый вариант обучения

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

writer = SummaryWriter('tensorboard_logs')  
num_epochs = 10

train_losses, train_accs, train_f1s = [], [], []
val_losses, val_accs, val_f1s = [], [], []

writer.add_hparams(
    {'lr': 0.001, 'batch_size': 32, 'epochs': num_epochs},
    {'hparam/train_acc': 0, 'hparam/val_acc': 0},
    run_name='base_experiment'
)

print(f'TensorBoard logs: ./tensorboard_logs')
for epoch in tqdm(range(num_epochs)):
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1, val_prec, val_rec = validate(model, test_loader, criterion, device)
    scheduler.step()

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_f1s.append(train_f1)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    val_f1s.append(val_f1)

    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Acc/train', train_acc, epoch)
    writer.add_scalar('F1/train', train_f1, epoch)
    writer.add_scalar('Loss/val', val_loss, epoch)
    writer.add_scalar('Acc/val', val_acc, epoch)
    writer.add_scalar('F1/val', val_f1, epoch)
    writer.add_scalar('Precision/val', val_prec, epoch)
    writer.add_scalar('Recall/val', val_rec, epoch)
    
    print(f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}')
    print(f'Val   Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}')

writer.close()

TensorBoard logs: ./tensorboard_logs


 10%|█         | 1/10 [01:24<12:40, 84.47s/it]

Train Loss: 0.3739, Acc: 0.8420, F1: 0.8420
Val   Loss: 0.4158, Acc: 0.8239, F1: 0.8198


 20%|██        | 2/10 [02:53<11:34, 86.86s/it]

Train Loss: 0.2920, Acc: 0.8777, F1: 0.8777
Val   Loss: 0.2213, Acc: 0.9214, F1: 0.9214


...

Train Loss: 0.0999, Acc: 0.9610, F1: 0.9610
Val   Loss: 0.1016, Acc: 0.9635, F1: 0.9635


 90%|█████████ | 9/10 [13:11<01:28, 88.63s/it]

Train Loss: 0.0910, Acc: 0.9649, F1: 0.9649
Val   Loss: 0.0962, Acc: 0.9642, F1: 0.9642


100%|██████████| 10/10 [14:41<00:00, 88.16s/it]

Train Loss: 0.0881, Acc: 0.9674, F1: 0.9674
Val   Loss: 0.0978, Acc: 0.9632, F1: 0.9632


In [6]:
test_writer = SummaryWriter('tensorboard_logs/test')
model.eval()
test_running_loss = 0.0
test_preds, test_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Final Test'):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

test_loss = test_running_loss / len(test_loader.dataset)
test_acc = accuracy_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, test_preds, average='weighted')
test_prec = precision_score(test_labels, test_preds, average='weighted', zero_division=0)
test_rec = recall_score(test_labels, test_preds, average='weighted', zero_division=0)

test_writer.add_hparams(
    {'lr': 0.001, 'batch_size': 32, 'epochs': num_epochs},
    {'hparam/test_acc': test_acc, 'hparam/test_f1': test_f1}
)
test_writer.add_scalar('Loss/test', test_loss, 0)
test_writer.add_scalar('Acc/test', test_acc, 0)
test_writer.add_scalar('F1/test', test_f1, 0)
test_writer.add_scalar('Precision/test', test_prec, 0)
test_writer.add_scalar('Recall/test', test_rec, 0)
test_writer.close()

print(f'Test Loss: {test_loss:.4f}, Acc: {test_acc:.4f}, F1: {test_f1:.4f}')

torch.save(model.state_dict(), 'model_base.pth')

Final Test: 100%|██████████| 125/125 [00:15<00:00,  7.82it/s]

Test Loss: 0.0978, Acc: 0.9632, F1: 0.9632


Вот тут надо не забывать развернуть MinIO через предоставленный docker-compose!

In [14]:
!docker compose up -d

time="2026-05-07T18:07:36+03:00" level=warning msg="c:\\Users\\gfgh\\Desktop\\HSE\\OZON\\HW4\\-tracking_homework_04\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Network tracking_homework_04_default  Creating
 Network tracking_homework_04_default  Created
 Container minio  Creating
 Container minio  Created
 Container minio  Starting
 Container minio  Started


Попингуем на всякий MinIO

In [15]:
import requests, time
url = "http://localhost:9000/minio/health/live"
print("Ожидание MinIO...")
for i in range(30):
    try:
        if requests.get(url).status_code == 200:
            print("MinIO готов!")
            break
    except:
        pass
    time.sleep(2)
else:
    print("MinIO не запустился")

Ожидание MinIO...
MinIO готов!


In [9]:
minio_client = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

bucket_name = 'models'

try:
    minio_client.head_bucket(Bucket=bucket_name)
except:
    minio_client.create_bucket(Bucket=bucket_name)
    print(f'Bucket "{bucket_name}" created.')

minio_client.upload_file('model_base.pth', bucket_name, 'model_base.pth')
print(f'Model model_base.pth uploaded to bucket "{bucket_name}".')

Bucket "models" created.
Model model_base.pth uploaded to bucket "models".


In [10]:
train2_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset-hw/Train_2/train.csv',
    root_dir='ai-vs-human-generated-dataset-hw/Train_2',
    transform=train_transform
)

test2_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset-hw/Test_2/test.csv',
    root_dir='ai-vs-human-generated-dataset-hw/Test_2',
    transform=test_transform
)

train2_loader = DataLoader(train2_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test2_loader = DataLoader(test2_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f'Train2 size: {len(train2_dataset)}, Test2 size: {len(test2_dataset)}')

Train2 size: 3997, Test2 size: 2000


Добавим дообучение!

In [11]:
def finetune_model(base_model_path=None, s3_download=True, bucket='models', key='model_base.pth'):
    model_ft = models.resnet18(pretrained=False)
    model_ft.fc = nn.Linear(model_ft.fc.in_features, 2)

    if s3_download:
        client = boto3.client(
            's3',
            endpoint_url='http://localhost:9000',
            aws_access_key_id='minioadmin',
            aws_secret_access_key='minioadmin',
            config=Config(signature_version='s3v4'),
            region_name='us-east-1'
        )
        client.download_file(bucket, key, 'temp_base.pth')
        state_dict = torch.load('temp_base.pth', map_location=device)
        os.remove('temp_base.pth')
    else:
        state_dict = torch.load(base_model_path, map_location=device)
    
    model_ft.load_state_dict(state_dict)
    model_ft = model_ft.to(device)

    optimizer_ft = optim.Adam(model_ft.parameters(), lr=0.0001)
    criterion_ft = nn.CrossEntropyLoss()
    scheduler_ft = optim.lr_scheduler.StepLR(optimizer_ft, step_size=3, gamma=0.1)

    writer_ft = SummaryWriter('finetune_logs')
    writer_ft.add_hparams(
        {'lr': 0.0001, 'batch_size': 32, 'epochs': 5, 'optimizer': 'Adam'},
        {'hparam/val_acc': 0, 'hparam/val_f1': 0},
        run_name='finetune_experiment'
    )
    
    epochs_ft = 5
    for epoch in range(epochs_ft):
        train_loss, train_acc, train_f1 = train_epoch(model_ft, train2_loader, criterion_ft, optimizer_ft, device)
        val_loss, val_acc, val_f1, _, _ = validate(model_ft, test2_loader, criterion_ft, device)
        scheduler_ft.step()
        
        writer_ft.add_scalar('Loss/train', train_loss, epoch)
        writer_ft.add_scalar('Acc/train', train_acc, epoch)
        writer_ft.add_scalar('F1/train', train_f1, epoch)
        writer_ft.add_scalar('Loss/val', val_loss, epoch)
        writer_ft.add_scalar('Acc/val', val_acc, epoch)
        writer_ft.add_scalar('F1/val', val_f1, epoch)
        
        print(f'Finetune Epoch {epoch+1}/{epochs_ft}: Train Loss {train_loss:.4f}, Val Acc {val_acc:.4f}')
    
    writer_ft.close()

    test_loss, test_acc, test_f1, test_prec, test_rec = validate(model_ft, test2_loader, criterion_ft, device)
    metrics = {
        'test_loss': test_loss,
        'test_accuracy': test_acc,
        'test_f1': test_f1,
        'test_precision': test_prec,
        'test_recall': test_rec
    }
    
    torch.save(model_ft.state_dict(), 'model_fine.pth')
    
    try:
        minio_client.upload_file('model_fine.pth', bucket, 'model_fine.pth')
        print('Fine-tuned model uploaded to S3 as model_fine.pth')
    except Exception as e:
        print(f'S3 upload failed: {e}')
    
    return metrics

In [12]:
finetune_metrics = finetune_model(
    base_model_path='model_base.pth',
    s3_download=True
)
print('Finetune metrics:')
print(json.dumps(finetune_metrics, indent=2))

c:\Users\gfgh\Desktop\HSE\VK_DL\ipynb_venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\gfgh\Desktop\HSE\VK_DL\ipynb_venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Validation: 100%|██████████| 63/63 [00:07<00:00,  8.02it/s]


Finetune Epoch 1/5: Train Loss 0.1321, Val Acc 0.9550


Validation: 100%|██████████| 63/63 [00:07<00:00,  8.13it/s]


Finetune Epoch 2/5: Train Loss 0.1252, Val Acc 0.9565


Validation: 100%|██████████| 63/63 [00:11<00:00,  5.59it/s]


...

Finetune Epoch 4/5: Train Loss 0.0942, Val Acc 0.9740


Validation: 100%|██████████| 63/63 [00:07<00:00,  8.09it/s]


Finetune Epoch 5/5: Train Loss 0.0884, Val Acc 0.9780


Validation: 100%|██████████| 63/63 [00:07<00:00,  8.16it/s]


Fine-tuned model uploaded to S3 as model_fine.pth
Finetune metrics:
{
  "test_loss": 0.07640496508777142,
  "test_accuracy": 0.978,
  "test_f1": 0.9779998679952477,
  "test_precision": 0.9780016920829121,
  "test_recall": 0.978
}


In [19]:
base_metrics = {
    'test_loss': test_loss,
    'test_accuracy': test_acc,
    'test_f1': test_f1,
    'test_precision': test_prec,
    'test_recall': test_rec
}
fine_metrics = finetune_metrics

print('=== Базовая модель (Test_1) ===')
for k, v in base_metrics.items():
    print(f'{k}: {v:.4f}')

print('\n=== Дообученная модель (Test_2) ===')
for k, v in fine_metrics.items():
    print(f'{k}: {v:.4f}')

=== Базовая модель (Test_1) ===
test_loss: 0.0978
test_accuracy: 0.9632
test_f1: 0.9632
test_precision: 0.9632
test_recall: 0.9632

=== Дообученная модель (Test_2) ===
test_loss: 0.0764
test_accuracy: 0.9780
test_f1: 0.9780
test_precision: 0.9780
test_recall: 0.9780


Сделаем выводы!

**Базовая модель** (Test_1) показала **accuracy 0.9632** и **F1 0.9632**, loss составил 0.0978.  
**Дообученная модель** (Test_2) улучшила все показатели: **accuracy вырос до 0.9780**, **F1 – до 0.9780**, а **loss снизился до 0.0764**. Precision и recall также повысились до 0.9780, демонстрируя сбалансированный прирост качества.

Относительно базовой версии **прирост accuracy и F1 составил ~1.5 п.п.**, а **ошибка уменьшилась на 22%** (с 0.0978 до 0.0764). Это подтверждает, что дообучение на втором датасете позволило модели стать точнее и увереннее в классификации AI- и human-изображений.

Теперь вынесем код в отдельные файлы и создадим DVC пайплайн!

In [16]:
!dvc init

ERROR: failed to initiate DVC - '.dvc' exists. Use `-f` to force.


In [17]:
!dvc stage add -n baseline -d ai-vs-human-generated-dataset-hw/Train_1 -d ai-vs-human-generated-dataset-hw/Test_1 -o model_base.pth -o base_metrics.json python train_stage.py

ERROR: Stage 'baseline' already exists in 'dvc.yaml'. Use '--force' to overwrite.


In [18]:
!dvc stage add -n finetune -d model_base.pth -d ai-vs-human-generated-dataset-hw/Train_2 -d ai-vs-human-generated-dataset-hw/Test_2 -o model_fine.pth -o finetune_metrics.json python finetune_stage.py

ERROR: Stage 'finetune' already exists in 'dvc.yaml'. Use '--force' to overwrite.


In [19]:
!dvc repro

Stage 'baseline' didn't change, skipping
Running stage 'finetune':
> python finetune_stage.py
Base model downloaded from S3.
Finetune Epoch 1/5 completed.
Finetune Epoch 2/5 completed.
Finetune Epoch 3/5 completed.
Finetune Epoch 4/5 completed.
Finetune Epoch 5/5 completed.
Finetuned model saved. Acc: 0.9690, F1: 0.9690
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.0+cu118
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
c:\Users\gfgh\Desktop\HSE\VK_DL\ipynb_venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\gfgh\Desktop\HSE\VK_DL\ipynb_venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [20]:
!dvc dag

+----------+ 
| baseline | 
+----------+ 
      *      
      *      
      *      
+----------+ 
| finetune | 
+----------+ 


In [21]:
!docker compose down

time="2026-05-07T18:11:48+03:00" level=warning msg="c:\\Users\\gfgh\\Desktop\\HSE\\OZON\\HW4\\-tracking_homework_04\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container minio  Stopping
 Container minio  Stopped
 Container minio  Removing
 Container minio  Removed
 Network tracking_homework_04_default  Removing
 Network tracking_homework_04_default  Removed
